In [2]:
!pip install langchain_community pypdf langchain_ollama langchain-chroma langchain_huggingface sentence_transformers langchain_core langchain-text-splitters


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter as cts

C:\Users\ytlil\PyCharmMiscProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
rag_data=PyPDFLoader("RAG.pdf").load()
len(rag_data)

1

In [5]:
#Split email into chunks
chunk_splitter=cts(separator="", chunk_size=50, chunk_overlap=10)
chunks=chunk_splitter.split_documents(rag_data)

In [6]:
#Load embedding model
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2452.98it/s]


In [7]:
#Create vector store
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="./rag_vectordb")

In [25]:
from langchain_core.prompts import ChatPromptTemplate as cpt
#from langchain_core.prompts.chat import SystemMessagePromptTemplate as spt,HumanMessagePromptTemplate as hpt
from langchain_ollama import ChatOllama
ollama_model=ChatOllama(model="phi3:3.8b ",temperature=0)

In [35]:
from typing import TypedDict, Optional, Dict, Any

class SupportState(TypedDict, total=False):
    email: Dict[str, Any]
    classification: str
    draft_response: Dict[str, Any]
    follow_up: Optional[Dict[str, Any]]

In [94]:

def node_classify_email(state: SupportState):
    global support_agent
    support_agent= workflow.compile()
    email_str=state["email"]["body"]+state["email"]["subject"]

    #Classification task
    system_message="""you are an expert customer support agent. Classify the email based on urgency as low, medium, high. Also classify them based on Topic for example billing, technical support, bug, feature request, etc.ALong with it suggest escalation is required or followup and why"""
    human_msg=f"Email content: {email_str}"
    classi_template = [
            ("system", system_message),
            ("human",human_msg)
        ]
    context_template = cpt.from_messages(classi_template)
    #message=context_template.format(query=query)
    #classification_response =ollama_model.invoke(message)
    classi_chain=context_template|ollama_model
    classification_response=classi_chain.invoke({"email_str":email_str})
    print("**Classification based on urgency and topic : **")
    print(classification_response.content)
    return {"classification": classification_response.content}

In [96]:
#Based on the classification, generate response to customer email
def node_generate_response(state: SupportState):
    context=""
    email_str=state["email"]["body"]+state["email"]["subject"]
    classification_response=state["classification"]
    #retrieve relevant chunks based on query and add to context
    ret_doc=vectorstore.similarity_search(query=email_str,k=2)
    for doc in ret_doc:
        context+=doc.page_content
    system_message="""you are an expert customer support agent. Based on the email content received, classification generated, and context from vector db - generate a response with following points :

    - reply text : reply mail to the customer email.

    - should escalate : true/false (escalation happens for complex issues that cannot be resolved with a single response.)

    - escalation summary : true/false , (and why) , along with short summary for human agent

    - follow up required : true/false
    """
    human_msg=f"Email content: {email_str} + knowledge: {context} + Classification: {classification_response}"
    chat_template = [
            ("system", system_message),
            ("human",human_msg)
        ]
    context_template = cpt.from_messages(chat_template)
    message=context_template.format(context=context, email_str=email_str, classification=classification_response)
    draft_resp=ollama_model.invoke(message)
    print("** printing response output**")
    print(draft_resp.content)
    draft_response={"text":draft_resp.content, "should_escalate": False, "follow_up_required": True}
    return {"draft_response": draft_response}

In [97]:
#Decision making - whether to escalate or auto respond
def node_escalate(state: SupportState):
    print("/n*****escalation ticket created****/n")

In [103]:
def node_follow_up(state: SupportState):
    email = state["email"]
    foruser = state["email"]["from_address"]
    followup = {"note": "for :" +foruser}
    print("Follow-up scheduled:", followup)
    return {"follow_up": followup}

In [99]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(SupportState)

# Add nodes
workflow.add_node("classify", node_classify_email)
workflow.add_node("respond", node_generate_response)
workflow.add_node("escalate", node_escalate)
workflow.add_node("follow_up", node_follow_up)

# Entry point
workflow.set_entry_point("classify")

# Normal flow
workflow.add_edge("classify", "respond")
def should_escalate(state: SupportState):
    return state["draft_response"].get("should_escalate", False)

workflow.add_conditional_edges(
    "respond",
    should_escalate,
    {
        True: "escalate",
        False: "follow_up"
    }
)
def needs_follow_up(state: SupportState):
    return state["draft_response"].get("follow_up_required", False)

workflow.add_conditional_edges(
    "follow_up",
    needs_follow_up,
    {
        True: END,
        False: END
    }
)

In [101]:
support_agent = workflow.compile()

In [105]:
result = support_agent.invoke({
    "email": {
        "subject": "feature crash",
        "body": "Export feature crashes when i click Export pdf button.",
        "from_address": "customer2@example.com"
    }
}, config={"reset": True})

result

**Classification based on urgency and topic : **
Classification of Urgency: High - The customer has encountered a critical issue that prevents them from completing their task, which in this case involves exporting data to PDF format. This could be seen as an urgent matter since it directly impacts the user's ability to work efficiently and effectively with your software or service.

Topic: Bug - The customer is experiencing a problem related to functionality within the application/service, which suggests that there might be an issue in need of investigation by technical support staff. 

Escalation Required: Yes - Given the urgency level (high) and potential impact on user productivity or satisfaction with your software/service, it is advisable for this email to receive immediate attention from a higher-level team member who can address such critical issues more effectively than standard support channels. This could involve escalating the issue within technical support if they are not a

{'email': {'subject': 'feature crash',
  'body': 'Export feature crashes when i click Export pdf button.',
  'from_address': 'customer2@example.com'},
 'classification': "Classification of Urgency: High - The customer has encountered a critical issue that prevents them from completing their task, which in this case involves exporting data to PDF format. This could be seen as an urgent matter since it directly impacts the user's ability to work efficiently and effectively with your software or service.\n\nTopic: Bug - The customer is experiencing a problem related to functionality within the application/service, which suggests that there might be an issue in need of investigation by technical support staff. \n\nEscalation Required: Yes - Given the urgency level (high) and potential impact on user productivity or satisfaction with your software/service, it is advisable for this email to receive immediate attention from a higher-level team member who can address such critical issues more 